In [12]:
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.readers.file import PDFReader
import os
import pandas as pd
from pathlib import Path

In [13]:
CSV_CANDIDATES = [
    Path("../Data/pdf_sample1/ChatbotData.csv"),
    Path("Data/pdf_sample1/ChatbotData.csv"),
    Path("../Data/ChatbotData.csv"),
    Path("Data/ChatbotData.csv"),
]
CSV_PATH = next((path for path in CSV_CANDIDATES if path.exists()), CSV_CANDIDATES[0])
cohere_api_key = os.environ["COHERE_API_KEY"]

Settings.llm = Cohere(
    model = 'command-r7b-12-2024',
    api_key = cohere_api_key,
    temperature = 0
)

Settings.embed_model = CohereEmbedding(
    api_key = cohere_api_key,
    model_name = 'embed-multilingual-v3.0',
    input_type = 'search_document',
    embed_batch_size = 96
)

print(f"CSV 경로 : {CSV_PATH.resolve()}")
print("Cohere / Llamaindex 설정 완료")

CSV 경로 : /Users/cheng80/Documents/WorkSpace/RAG/Data/pdf_sample1/ChatbotData.csv
Cohere / Llamaindex 설정 완료


#### CSV를 문서 형태로 변환

In [14]:
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [15]:
TEXT_COLUMNS = ['Q', 'A']
METADATA_COLUMNS = ['label']

MAX_ROWS = 1000
df = df.head(MAX_ROWS).copy()

display(df.head())
print('문서와 대상 컬럼 :', TEXT_COLUMNS)
print('메타데이터 컬럼 :', METADATA_COLUMNS)
print('사용할 행 수 :', len(df))

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼 : ['Q', 'A']
메타데이터 컬럼 : ['label']
사용할 행 수 : 1000


In [16]:
# 각 Row를 질문-답변 형태의 문서로 변환

def row_to_document(row: pd.Series, row_number: int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue

        text_parts.append(f"{column}: {value}")

    metadata = {
        "row_number" : row_number,
        "label" : row['label']
    }

    return Document(
        text = " | ".join(text_parts),
        metadata = metadata
    )

# DataFrame의 각 row를 Document로 변환
documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print("생성된 Document 수 :", len(documents))
print("첫번째 Document 예제")
print(documents[0].text)

생성된 Document 수 : 1000
첫번째 Document 예제
Q: 12시 땡! | A: 하루가 또 가네요.


In [17]:
# 문서목록으로 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

# 검색된 문서를 바탕으로 답변하는 chat engine을 제작
chat_engine = index.as_chat_engine(
    chat_mode='context',
    similarity_top_k = 5,
    verbose = True
)

print("챗봇 준비 완료")

2026-04-28 12:10:26,003 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:26,704 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:27,370 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:28,027 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:28,767 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:29,425 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:30,115 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:30,920 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:31,566 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:32,217 - INFO - HTTP Request: POST https://api.cohere.com/v2/embe

챗봇 준비 완료


In [18]:
# Test
question = "12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?"
response = chat_engine.chat(question)
print("질문 :", question)
print("응답 :")
print(response)

2026-04-28 12:10:33,398 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 12:10:34,293 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


질문 : 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답 :
"12시 땡! 이라는 질문에는 "하루가 또 가네요."라는 답변이 연결되어 있습니다.


In [19]:
# 여러 번 질문하고
# exit, quit 종료

while True:
    user_question = input("질문을 입력하세요 : ").strip()
    if user_question.lower() in {'exit', 'quit'}:
        print("챗봇을 종료합니다.")
        break
    
    if not user_question:
        print("빈 질문은 처리할 수 없습니다. 다시 입력해주세요")
        continue

    answer = chat_engine.chat(user_question)
    print("\n[응답]")
    print(answer)
    print("-" * 60)

챗봇을 종료합니다.
